**`harmonize_transactions`**

Harmonize transaction records into `US_transaction-spine-2026` and link each sale to its parcel(s), via `parcel_id_local` (NC/WI) or `address_id_local` (MA, which carries no shared parcel id).

# Configure

In [ ]:
import argparse

from openplaces.io.harmonizer import Harmonizer

In [ ]:
parser = argparse.ArgumentParser(
    description='Harmonize transaction records and link them to parcels'
)
parser.add_argument(
    '--recipe_id',
    help='Harmonization recipe (e.g. "US_transaction-spine-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to harmonize (e.g. "US-WI-DA")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess admin IDs even if output already exists',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--recipe_id US_transaction-spine-2026 '
    # Dane, Oneida, and Vilas counties, WI (parcel_id_local linkage)
    '--admin_ids US-WI-DA US-WI-ON US-WI-VI '
    # Middlesex county, MA (address_id_local linkage; covers Somerville,
    # Cambridge, Medford, ...)
    'US-MA-MI '
    '--reprocess '
    '--verbose'
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

In [ ]:
# Show recipe parameters
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

pretty_print(get_recipe_by_id(args.recipe_id))

# Run

In [ ]:
harmonizer = Harmonizer(args.recipe_id, args.admin_ids, verbose=args.verbose)

In [ ]:
harmonizer.harmonize(reprocess=args.reprocess)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

## Match-rate summary by county and source

A transaction counts as "linked" if any parcel attribute the recipe's `link_by_id` steps attach (`use_group`, `land_value`, ...) is non-null -- these only ever get filled in by a successful join, whichever key (`parcel_id_local` or `address_id_local`) that source actually used.

In [ ]:
import pandas as pd

from openplaces import get_entities

LINKED_COLUMNS = [
    'use_group',
    'use_subgroup',
    'land_value',
    'improvement_value',
    'year_built',
    'n_dwellings',
]

In [ ]:
rows = []
for admin_id in args.admin_ids:
    spine = get_entities(args.recipe_id, admin_id)
    # Which of LINKED_COLUMNS are actually present varies by admin unit: a
    # link_by_id step only adds columns its reference recipe actually has
    # (e.g. wiscedu has no use_subgroup/year_built/n_dwellings, so WI rows
    # never get those columns at all, not even as all-null).
    present_columns = [c for c in LINKED_COLUMNS if c in spine.columns]
    is_linked = spine[present_columns].notna().any(axis=1)
    for source, group in spine.groupby('source'):
        rows.append(
            {
                'admin_id': admin_id,
                'source': source,
                'n_transactions': len(group),
                'n_linked': int(is_linked.loc[group.index].sum()),
            }
        )

summary = pd.DataFrame(rows)
summary['match_rate'] = summary['n_linked'] / summary['n_transactions']
summary

In [ ]:
summary.set_index('admin_id')['match_rate'].plot(
    kind='bar', ylim=(0, 1), figsize=(8, 3), title='Transaction → parcel match rate'
)

## Linked vs. unlinked examples

In [ ]:
args.admin_ids

In [ ]:
spine = get_entities(args.recipe_id, args.admin_ids)
present_columns = [c for c in LINKED_COLUMNS if c in spine.columns]
is_linked = spine[present_columns].notna().any(axis=1)

linked_sample = spine[is_linked].sample(min(5, is_linked.sum()))
linked_sample.T

In [ ]:
spine['admin3_id'].value_counts()

In [ ]:
unlinked = spine[~is_linked]
unlinked_sample = unlinked.sample(min(5, len(unlinked)))
pd.options.display.max_rows = len(unlinked.columns)
unlinked_sample.T